## Creating Spark Session

In [1]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

packages = ["org.apache.hadoop:hadoop-aws:3.4.0"]

builder = (
    SparkSession.builder
    .appName("MinIO-Delta-Lake")
    .master("local[*]")
    .config("spark.hadoop.fs.s3a.endpoint", "http://127.0.0.1:9000")
    .config("spark.hadoop.fs.s3a.endpoint.region", "us-east-1")
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.secret.key", "miniopass")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(
    builder,
    extra_packages=packages
).getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/18 15:40:31 WARN Utils: Your hostname, TIC-06, resolves to a loopback address: 127.0.1.1; using 172.25.124.25 instead (on interface eth0)
26/11/18 15:40:31 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/realdo/.cache/pypoetry/virtualenvs/projeto-ed-satc-Gz-K1ou1-py3.11/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/realdo/.ivy2.5.2/cache
The jars for the packages stored in: /home/realdo/.ivy2.5.2/jars
io.delta#delta-spark_2.13 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-3b8e8a6e-1bf7-4b2d-a276-e44e58934c0e;1.0
	confs: [default]
	found io.delta#delta-spark_2.13;4.0.0 in central
	found io.delta#delta-storage;4.0.0 in central
	found org.antlr#antlr4-runtime;4

## Reading Bronze Tables

In [2]:
caminho_bronze = 's3a://bronze'

df_categoria_produto = spark.read.format("delta").load(f"{caminho_bronze}/categoria_produto")
df_inpc_valor = spark.read.format("delta").load(f"{caminho_bronze}/inpc_valor")
df_ipca_valor = spark.read.format("delta").load(f"{caminho_bronze}/ipca_valor")
df_municipio = spark.read.format("delta").load(f"{caminho_bronze}/municipio")
df_pib_municipio = spark.read.format("delta").load(f"{caminho_bronze}/pib_municipio")
df_pnad_domicilio = spark.read.format("delta").load(f"{caminho_bronze}/pnad_domicilio")
df_pnad_morador = spark.read.format("delta").load(f"{caminho_bronze}/pnad_morador")
df_pnad_ocupacao = spark.read.format("delta").load(f"{caminho_bronze}/pnad_ocupacao")
df_populacao_municipio = spark.read.format("delta").load(f"{caminho_bronze}/populacao_municipio")
df_producao_agricola = spark.read.format("delta").load(f"{caminho_bronze}/producao_agricola")
df_producao_origem_animal = spark.read.format("delta").load(f"{caminho_bronze}/producao_origem_animal")
df_rebanho_municipio = spark.read.format("delta").load(f"{caminho_bronze}/rebanho_municipio")

26/11/18 15:40:45 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


## Adding time and file name as columns

In [3]:
from pyspark.sql.functions import current_timestamp, lit

df_categoria_produto = df_categoria_produto.withColumn("data_hora_silver", current_timestamp()).withColumn("nome_tabela", lit("categoria_produto"))
df_inpc_valor = df_inpc_valor.withColumn("data_hora_silver", current_timestamp()).withColumn("nome_tabela", lit("inpc_valor"))
df_ipca_valor = df_ipca_valor.withColumn("data_hora_silver", current_timestamp()).withColumn("nome_tabela", lit("ipca_valor"))
df_municipio = df_municipio.withColumn("data_hora_silver", current_timestamp()).withColumn("nome_tabela", lit("municipio"))
df_pib_municipio = df_pib_municipio.withColumn("data_hora_silver", current_timestamp()).withColumn("nome_tabela", lit("pib_municipio"))
df_pnad_domicilio = df_pnad_domicilio.withColumn("data_hora_silver", current_timestamp()).withColumn("nome_tabela", lit("pnad_domicilio"))
df_pnad_morador = df_pnad_morador.withColumn("data_hora_silver", current_timestamp()).withColumn("nome_tabela", lit("pnad_morador"))
df_pnad_ocupacao = df_pnad_ocupacao.withColumn("data_hora_silver", current_timestamp()).withColumn("nome_tabela", lit("pnad_ocupacao"))
df_populacao_municipio = df_populacao_municipio.withColumn("data_hora_silver", current_timestamp()).withColumn("nome_tabela", lit("populacao_municipio"))
df_producao_agricola = df_producao_agricola.withColumn("data_hora_silver", current_timestamp()).withColumn("nome_tabela", lit("producao_agricola"))
df_producao_origem_animal = df_producao_origem_animal.withColumn("data_hora_silver", current_timestamp()).withColumn("nome_tabela", lit("producao_origem_animal"))
df_rebanho_municipio = df_rebanho_municipio.withColumn("data_hora_silver", current_timestamp()).withColumn("nome_tabela", lit("rebanho_municipio"))

## Defining Rules to the Column Names

In [4]:
values_columns = [
    ["ID", "IDENTIFIER"],
    ["UF", "UNIDADE_DA_FEDERACAO"],
    ["CAPITAL", "MINICIPIO_DA_CAPITAL"],
    ["RM_RIDE", "REGIAO_METROPOLITANA_E_ADMINISTRATIVA_INTEGRADA_DE_DESENVOLVIMENTO"],
    ["UPA", "UNIDADE_PRIMARIA_DE_AMOSTRAGEM"],
    ["V1008", "NUMERO_DE_SELECAO_DO_DOMICILIO"],
    ["V1014", "PAINEL"],
    ["V1022", "SITUACAO_DO_DOMICILIO"],
    ["V1023", "TIPO_DE_AREA"],
    ["V2001", "NUMERO_DE_PESSOAS_NO_DOMICILIO"],
    ["V2003", "NUMERO_DE_ORDEM"],
    ["V2005", "CONDICAO_NO_DOMICILIO"],
    ["V2007", "SEXO"],
    ["V2009", "IDADE_DO_MORADOR_NA_DATA_DE_REFERENCIA"],
    ["V2010", "COR_OU_RACA"],
    ["V3001", "SABE_LER_E_ESCREVER"],
    ["V3002", "FREQUENTA_ESCOLA"],
    ["V3009A", "CURSO_MAIS_ELEVADO_FREQUENTADO"],
    ["VD3004", "NIVEL_DE_INSTRUCAO_MAIS_ELEVADO_ALCANCADO"],
    ["VD3005", "ANOS_DE_ESTUDO"],
    ["VD4001", "FORCA_DE_TRABALHO_NA_SEMANA_PARA_APLICAVEIS"],
    ["VD4002", "OCUPACAO_NA_SEMANA_PARA_APLICAVEIS"],
    ["V1027", "PESO_DO_DOMICILIO_E_DAS_PESSOAS_SEM_CALIBRACAO"],
    ["V1028", "PESO_DO_DOMICILIO_E_DAS_PESSOAS_COM_CALIBRACAO"],
    ["V1029", "PROJECAO_DA_POPULACAO_POR_NIVEIS_GEOGRAFICOS"],
    ["V4009", "QUANTOS_TRABALHOS_POR_SEMANA"],
    ["V4010", "CODIGO_DA_OCUPACAO"],
    ["V4012", "TIPO_DE_TRABALHO"],
    ["V4013", "CODIGO_ATIVIDADE_PRINCIPAL_DO_NEGOCIO_EMPRESA"],
    ["V4029", "TRABALHO_COM_CARTEIRA_DE_TRABALHO_ASSINADA"],
    ["V4032", "TRABALHO_ERA_CONTRIBUINTE_DE_INSTITUTO_DE_PREVEDENCIA"],
    ["V4039", "HORAS_TRABALHADAS_POR_SEMANA_NO_TRABALHO_PRINCIPAL"],
    ["VD4009", "POSICAO_NA_OCUPACAO_E_CATEGORIA_NO_TRABALHO_PRINCIPAL_PARA_APLICAVEIS"],
    ["VD4010", "GRUPAMENTOS_DE_ATIVADADE_NO_TRABALHO_PRINCIPAL_PARA_APLICAVEIS"],
    ["VD4011", "GRUPAMENTOS_OCUPACIONAIS_NO_TRABALHO_PRINCIPAL_PARA_APLICAVEIS"],
    ["VD4012", "CONTRIBUICAO_PARA_INSTITUTO_DE_PREVIDENCIA_PARA_APLICAVEIS"],
    ["VD4016", "RENDIMENTO_MENSAL_HABITUAL_NO_TRABALHO_PRINCIPAL_PARA_APLICAVEIS"],
    ["VD4017", "RENDIMENTO_MENSAL_EFETIVO_NO_TRABALHO_PRINCIPAL_PARA_APLICAVEIS"],
    ["VD4019", "RENDIMENTO_MENSAL_HABITUAL_EM_TODOS_OS_TRABALHOS_PARA_APLICAVEIS"],
    ["VD4020", "RENDIMENTO_MENSAL_EFETIVO_EM_TODOS_OS_TRABALHOS_PARA_APLICAVEIS"],
    ["VD4031", "HORAS_TRABALHADAS_HABITUAL_EM_TODOS_OS_TRABALHOS_PARA_APLICAVEIS"],
]


In [5]:
def apply_name_rules(colname: str) -> str:
    n = colname.upper()
    for columns in values_columns:
        n = n.replace(columns[0], columns[1])
    return n

def safe_drop(df, cols):
    existing = set(df.columns)
    to_drop = [c for c in cols if c in existing]
    return df.drop(*to_drop) if to_drop else df

## Applying The Column Name Rules and Saving

In [6]:
dataframes = [
    df_categoria_produto, df_inpc_valor, df_ipca_valor, df_municipio, df_pib_municipio,df_pnad_domicilio,
    df_pnad_morador, df_pnad_ocupacao, df_populacao_municipio, df_producao_agricola, df_producao_origem_animal,
    df_rebanho_municipio
]
caminho_silver = 's3a://silver'

for currDf in dataframes:
    new_cols = [apply_name_rules(c) for c in currDf.columns]
    currDf = currDf.toDF(*new_cols)

    currDf = safe_drop(currDf, ["DATA_HORA_BRONZE", "NOME_ARQUIVO"])

    currDf.write.format("delta").mode("overwrite").save(f"{caminho_silver}/"+currDf.select("NOME_TABELA").first()["NOME_TABELA"])

26/11/18 15:40:56 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/11/18 15:41:20 WARN S3ABlockOutputStream: Application invoked the Syncable API against stream writing to categoria_produto/part-00000-98208d3e-3e1a-4d79-83a4-51593f54f06d-c000.snappy.parquet. This is Unsupported
                                                                                